# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.1/117.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 34.0 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 9.8 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# login

In [ ]:
import huggingface_hub
huggingface_hub.login(Hugging_Face_TOKEN)

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import random
import pandas as pd
import re
import string
import sys
import argparse
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
from sklearn.utils import shuffle
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split

import emoji
import pyarabic.araby as araby

In [ ]:
import torch
import torch.distributed as dist

# Define Evaluation Function

In [ ]:
def evaluate(y_true, y_pred):
    labels = ['إيجابية', 'سلبية', 'محايدة']
    mapping = {'محايدة': 0,
               'Neutral':0,
               'neutral':0,
               'سلبية': 1,
               'negative':1,
               'Negative':1,
               'إيجابية':2,
               'positive':2,
               'Positive':2,
               'none':3}
    def map_func(x):
        return mapping.get(x, 1)

    y_true = np.vectorize(map_func)(y_true)
    y_pred = np.vectorize(map_func)(y_pred)

    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    print(f'Accuracy: {accuracy:.3f}')

    # Generate accuracy report
    unique_labels = set(y_true)  # Get unique labels

    f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
    print('\nf1_score: ', f1t)

    # Generate classification report
    class_report = classification_report(y_true=y_true,
                                         y_pred=y_pred,
                                         digits=4,
                                         target_names=['Neutral', 'Negative', 'Positive', 'Unclassified'])
    print('\nClassification Report:')
    print(class_report)

    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=[0, 1, 2, 3])
    print('\nConfusion Matrix:')
    print(conf_matrix)

# Define Predict Function

In [ ]:
def predict(X_test, pipe):
    y_pred = []
    # Create a Hugging Face Dataset
    dataset = Dataset.from_pandas(X_test)

    # Use the pipeline with a dataset
    outputs = pipe(dataset["text"], batch_size=32,
                   pad_token_id=pipe.tokenizer.eos_token_id)

    # Process results
    results = [output[0]['generated_text'].lower() for output in outputs]

    # Extract the text after "answer:" for each result
    for i in tqdm(range(len(results))):
        answer_index = results[i].find("answer:") + len("answer:")
        extracted_text = results[i][answer_index:].strip()
        if ('إيجابي' or 'positive' or 'Positive' or 'pos') in extracted_text:
            y_pred.append("إيجابية")
        elif ('سلبي' or 'negative' or 'Negative' or 'neg') in extracted_text:
            y_pred.append("سلبية")
        elif ('محايد' or 'Neutral' or 'neutral' or 'neu') in extracted_text:
            y_pred.append("محايدة")
        else:
            y_pred.append("none")
    return y_pred

# Load the Data

In [ ]:
data1 = pd.read_csv('train_all.csv')
ind = pd.read_excel('Used ASAD For Training.xlsx')

data = data1.loc[ind['Unnamed: 0'].values]
data = data[["Text", "sentiment"]]

data.columns = ["Text", "sentiment"]
print(data["sentiment"].value_counts())

sentiment
neutral     16289
negative     3890
positive     3778
Name: count, dtype: int64


# Preprocessing

In [ ]:
data['Text'] = data['Text'].str.replace(r'[^\w\s]+', '')
data['Text'] = data['Text'].str.replace("\s+", " ", regex=True)

data.head()

,Text,sentiment
48348,ياحبّي المغرور ياللي دِفاك شعور رد القمر للنور...,positive
47523,من شدة حبك له تتمنى لو إن الأسى يأكلك بأكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا أمتلك تعريفا واضحا للإهاب بس طالما جامعة ال...,neutral


In [ ]:
arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("گ", "ك", text)
    return text

In [ ]:
data['Text'] = data['Text'].apply(normalize_arabic)
data['Text'] = data['Text'].apply(remove_repeating_char)

data.head()

,Text,sentiment
48348,ياحبّي المغرور ياللي دِفاك شعور رد القمر للنور...,positive
47523,من شدة حبك له تتمنى لو ان الاسى ياكلك باكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا امتلك تعريفا واضحا للاهاب بس طالما جامعة ال...,neutral


In [ ]:
def data_cleaning(text):
    """Clean and preprocess text data.
    Args:
        text (pd.Series): A pandas Series containing text data to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text data.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Remove diacritics.
    - Remove English.
    - fix elongated words
    - Normalize Characters
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Remove '_' symbol from text
    pattern = re.compile('_')
    clean = clean.apply(lambda r: re.sub(pattern, ' ', r))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Remove emojis
    clean = clean.apply(lambda r: emoji.replace_emoji(r, replace=""))
    # Remove diacritics
    clean = clean.apply(lambda r: araby.strip_diacritics(r))
    # Remove English
    # pattern = re.compile(r'[a-zA-Z]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Fix elongated words
    pattern = re.compile(r'(.)\1{2,}')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=r'\1'))
    # Normalize letters
    pattern = re.compile("[إأآا]")
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl="ا"))
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))  # Remove extra whitespaces between words

    return clean

In [ ]:
data['Text'] = data_cleaning(data['Text'])

In [ ]:
import re

def remove_specific_term(text):
  cleaned_text = re.sub(r'\bpictwitercom\w*\b', '', text)
  cleaned_text = cleaned_text.strip() # Remove any leading or trailing whitespace
  return cleaned_text

data['Text'] = data['Text'].apply(remove_specific_term)
data.head()

,Text,sentiment
48348,ياحبي المغرور ياللي دفاك شعور رد القمر للنور و...,positive
47523,من شدة حبك له تتمنى لو ان الاسى ياكلك باكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا امتلك تعريفا واضحا للاهاب بس طالما جامعة ال...,neutral


In [ ]:
def remove_ids(text):
  return text.split("—")[0].strip()

data['Text'] = data['Text'].apply(remove_ids)
data.head()

,Text,sentiment
48348,ياحبي المغرور ياللي دفاك شعور رد القمر للنور و...,positive
47523,من شدة حبك له تتمنى لو ان الاسى ياكلك باكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا امتلك تعريفا واضحا للاهاب بس طالما جامعة ال...,neutral


In [ ]:
# Mapping dictionary
sentiment_map = {
    'positive': 'إيجابية',
    'negative': 'سلبية',
    'neutral': 'محايدة'
}

# Apply the mapping
data['sentiment'] = data['sentiment'].map(sentiment_map)
data.head()

,Text,sentiment
48348,ياحبي المغرور ياللي دفاك شعور رد القمر للنور و...,إيجابية
47523,من شدة حبك له تتمنى لو ان الاسى ياكلك باكملك و...,إيجابية
2695,يالليل يا جامع على الود قلبين,محايدة
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,إيجابية
7247,لا امتلك تعريفا واضحا للاهاب بس طالما جامعة ال...,محايدة


In [ ]:
data.drop_duplicates(subset='text', inplace = True)
data.dropna(inplace = True)
data.shape

(23957, 2)

In [ ]:
data = data.reset_index(drop = True)

# Split Data

In [ ]:
indecies = pd.read_csv('Train-Val-Test-Indecies-Climate-ASAD.csv')

# Clean and convert to integer index arrays
test = data.loc[indecies['test'].dropna().astype(int).values]
train = data.loc[indecies['train'].dropna().astype(int).values]
val = data.loc[indecies['val'].dropna().astype(int).values]

In [ ]:
train = train.reset_index(drop=True)

def generate_prompt(data_point):
    return f"""
    [INST] أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر.
    قم بتصنيف مشاعر الجملة العربية التالية  بين قوسين مربعين على اليسار بناءً على نبرتها العاطفية. اختر مشاعر واحدة فقط من بين: إيجابية، سلبية، أو محايدة لهذه الجملة العربية.  [/INST]
    [{data_point["Text"]}] = [{data_point["sentiment"]}]
            """.strip()

def generate_test_prompt(data_point):
    return f"""
    أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر.
    قم بتصنيف مشاعر الجملة العربية التالية بين قوسين مربعين على اليسار بناءً على نبرتها العاطفية. اختر مشاعر واحدة فقط من بين: إيجابية، سلبية، أو محايدة لهذه الجملة العربية.
    [{data_point["Text"]}] =
            """.strip()

train = pd.DataFrame(train.apply(generate_prompt, axis=1),
                       columns=["Text"])
val = pd.DataFrame(val.apply(generate_prompt, axis=1),
                      columns=["Text"])

In [ ]:
y_true = test["sentiment"]
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1), columns=["Text"])

train_data = Dataset.from_pandas(train)
eval_data = Dataset.from_pandas(train)

# Load Model

In [ ]:
import os

os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"
os.environ["PYTORCH_USE_CUDA_DSA"] = "1"

model_name = "core42/jais-13b"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map = {"": 0},
    quantization_config=bnb_config,
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          use_fast=False,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

configuration_jais.py:   0%|          | 0.00/6.76k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/core42/jais-13b:
- configuration_jais.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_jais.py:   0%|          | 0.00/68.6k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/core42/jais-13b:
- modeling_jais.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


# Predict Without Fine-tuning

In [ ]:
max_length = tokenizer.model_max_length
for i in tqdm(range(len(X_test[:5]))):
    prompt = X_test.iloc[i]["Text"]
    result = pipe(prompt, truncation = True, pad_token_id=tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("=")[-1].lower()
    print(answer)

 20%|██        | 1/5 [00:03<00:14,  3.52s/it]

 إيجابي (إيجابي)
    [الله يرحمه ويغمد روحه الجنة تعازينا لاهل السلطنة


 40%|████      | 2/5 [00:06<00:08,  2.95s/it]

 سلبي
    [@ care السلام عليكم لو تكرمت انا يظهر لي في ابشر اعمال منشاة باسمي


 60%|██████    | 3/5 [00:08<00:05,  2.73s/it]


    <br>
    <br>
    <br>
    <br>
    <br>


 80%|████████  | 4/5 [00:11<00:02,  2.65s/it]

 [إيجابي]
    [شكلي بحاول بكثر مااقدر اروح ماط


100%|██████████| 5/5 [00:13<00:00,  2.71s/it]


    [إيجابي، سلبي، محايد]
    [- (وعسى ان تكرهوا شيئا وهو


# Fine-tune Model

In [ ]:
from trl import SFTConfig

lora_target_modules = ["c_attn", "c_proj", "c_fc"]

peft_config = LoraConfig(
    lora_alpha=256,
    lora_dropout=0.05,
    r=128,
    target_modules=lora_target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

training_arguments = SFTConfig(
    output_dir="logs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1, # 4
    optim="paged_adamw_32bit",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=25,
    learning_rate=5e-5,
    weight_decay=0,
    fp16=False,
    bf16=False,
    max_grad_norm=1,
    max_steps=-1,
    warmup_ratio=0.1,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    eval_strategy="epoch",
    dataset_text_field="Text",
    max_seq_length=128
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments
)

Adding EOS to train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained("/content/drive/MyDrive/Jais")

Epoch,Training Loss,Validation Loss
1,1.112100,0.843335
2,0.720800,0.596313
3,0.582600,0.475231


# Predict

In [ ]:
from peft import PeftModel, PeftConfig
# Load the Lora model
model = PeftModel.from_pretrained(model, "/content/drive/MyDrive/Jais")

In [ ]:
for i in tqdm(range(len(X_test[:5]))):
    prompt = X_test.iloc[i]["Text"]
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to("cuda")
    result = model.generate(
        **inputs,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.2,
        max_new_tokens=20,
        pad_token_id=tokenizer.pad_token_id,
        do_sample = True
    )
    decoded = tokenizer.decode(result[0], skip_special_tokens=True)
    answer = decoded.split("=")[-1].lower()
    print(answer)

 20%|██        | 1/5 [00:03<00:15,  3.79s/it]

 [@ السلام عليكم ورحمة الله وبركاته هل صحيح ان


 40%|████      | 2/5 [00:07<00:10,  3.64s/it]

 [محايدة]
    [انا موظف حكومي تابع لوزارة الشؤون البلدية والقروية هل يحق للشركة التي


 60%|██████    | 3/5 [00:10<00:07,  3.56s/it]

 [محايدة] [@ السلام عليكم


 80%|████████  | 4/5 [00:14<00:03,  3.59s/it]

 [محايدة] 
    [@ care السلام عليكم هل يحق للمنشاة اجبار الموظف المدعوم للعودة للعمل


100%|██████████| 5/5 [00:17<00:00,  3.57s/it]

 [محايدة] [@ care السلام عليكم ورحمة الله وبركاته هل يحق للمنشاة اجبار الموظف المدعوم


In [ ]:
y_pred = []

for i in tqdm(range(len(X_test))):
    prompt = X_test.iloc[i]["Text"]
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to("cuda")
    result = model.generate(
        **inputs,
        top_p=0.9,
        max_new_tokens=5,
        temperature=0.2,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.2,
        do_sample=True,
    )
    decoded = tokenizer.decode(result[0], skip_special_tokens=True)
    answer = decoded.split("=")[-1].lower()
    # print(answer)

    if ('إيجابي' or 'positive' or 'Positive' or 'pos' or 'إيجابيّة' or 'إيجابية') in answer:
        y_pred.append("إيجابية")
    elif ('سلبي' or 'negative' or 'Negative' or 'neg') in answer:
        y_pred.append("سلبية")
    elif ('محايد' or 'Neutral' or 'neutral' or 'neu') in answer:
        y_pred.append("محايدة")
    else:
        y_pred.append("none")

100%|██████████| 3594/3594 [1:01:06<00:00,  1.02s/it]


In [ ]:
evaluate(y_true, y_pred)

Accuracy: 0.757

f1_score:  0.7593062369962863

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.8296    0.9050    0.8657      2443
    Negative     0.7227    0.2945    0.4185       584
    Positive     0.7167    0.5979    0.6519       567
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.7574      3594
   macro avg     0.5673    0.4494    0.4840      3594
weighted avg     0.7944    0.7574    0.7593      3594


Confusion Matrix:
[[2211   58  120   54]
 [ 252  172   14  146]
 [ 202    8  339   18]
 [   0    0    0    0]]


# Pred on Collected data

In [ ]:
collected = pd.read_excel('All Climate Change Data - All Related.xlsx')

collected.drop_duplicates(subset='text', inplace = True)
collected.dropna(inplace = True, subset='text')
collected.reset_index(drop=True, inplace = True)

In [ ]:
collected['Final Label'] = collected['Final Label'].map({
    'Negative': 'سلبية',
    'Neutral': 'محايدة',
    'Positive': 'إيجابية'
})

In [ ]:
collected['text'] = collected['text'].str.replace(r'[^\w\s]+', '')
collected['text'] = collected['text'].str.replace("\s+", " ", regex=True)

collected['text'] = collected['text'].apply(normalize_arabic)
collected['text'] = collected['text'].apply(remove_repeating_char)

collected['text'] = data_cleaning(collected['text'])

collected['text'] = collected['text'].apply(remove_ids)

collected.dropna(inplace = True)
collected = collected.drop_duplicates(subset = 'text')
collected = collected.rename(columns={'text': 'Text'})

collected_test = pd.DataFrame(collected.apply(generate_test_prompt, axis=1), columns=["Text"])

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(collected_test))):
    prompt = collected_test.iloc[i]["Text"]
    result = pipe(prompt, truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("=")[-1].lower()

    if ("سلبية" in answer
        or "سلبي" in answer
        or "سلبيّة" in answer):
        pred.append("سلبية")
    elif ("إيجابية" in answer
          or "إيجابي" in answer
          or "إيجابيّة" in answer):
        pred.append("إيجابية")
    elif ("محايدة" in answer
          or "محايد" in answer):
        pred.append("محايدة")
    else:
        pred.append("none")

100%|██████████| 8992/8992 [8:50:48<00:00,  3.54s/it]


In [ ]:
evaluate(collected['Final Label'], pred)

Accuracy: 0.212

f1_score:  0.23509538071358313

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.4216    0.4360    0.4287      3752
    Negative     0.7849    0.0481    0.0907      3035
    Positive     0.6058    0.0571    0.1044      2205
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.2122      8992
   macro avg     0.4531    0.1353    0.1560      8992
weighted avg     0.5894    0.2122    0.2351      8992


Confusion Matrix:
[[1636   31   60 2025]
 [1148  146   22 1719]
 [1096    9  126  974]
 [   0    0    0    0]]
